# Adaptive Residual Steering (ARS) — Kaggle Execution Runner
## V9 — Two Parallel Experiments

### Experiment A (this notebook): Big SteerNet + On-Policy
- **rank 128**, 10 layers, rho_max 0.30, on-policy k=4
- Pipeline: `S1 (with on-policy) → S5 → Bench`
- Time: ~70 min

### Experiment B (copy of this notebook): S1 + STaR (S1R)
- Change config: `RUN_STAR = True`, `ON_POLICY = False`
- Pipeline: `S1 → S1R (STaR) → S5 → Bench`
- Time: ~3-5 hours (generation heavy)

## 1. Environment & Hardware Check

In [ ]:
# Set CUDA allocator config BEFORE any torch/transformers import.
# expandable_segments:True lets the allocator satisfy fragmented-memory requests
# without requiring a single contiguous free block — critical for 10-hook training.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!nvidia-smi
!pip install -q "transformers>=4.36.0" "datasets>=2.16.0" "accelerate>=0.25.0" matplotlib pandas scipy tqdm


## 2. Clone Repository (`development` branch)

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"
import sys

REPO_DIR = "/kaggle/working/adaptive-residual-steering"
BRANCH = "development"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/Abdullah182155/adaptive-residual-steering.git {REPO_DIR}
else:
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 3. Per-Layer Norm Calibration (V5)

In [ ]:
import os
os.chdir(REPO_DIR)
print(f"Current directory: {os.getcwd()}")

# V9: ALL 10 candidate layers — bigger SteerNet
NEW_LAYERS = '8 11 13 15 17 19 20 21 22 24'
print(f"Calibrating norms for 10-layer route: [{NEW_LAYERS}]")
!python scripts/calibrate_norms.py \
    --n_samples 5000 \
    --output norms_calibration.json \
    --candidate_layers {NEW_LAYERS}

print("\n✅ Norm calibration complete:")
!ls -lh norms_calibration.json

## 4. Verify Codebase Integrity & Unit Tests

In [ ]:
import os
import sys
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"Current directory: {os.getcwd()}")
print(f"Tests directory exists: {os.path.exists('tests/')}")
!pytest tests/ -q --tb=short

## 5. Architecture Configuration

In [ ]:
# v5: rank-64 with causal convolution and per-layer caps
# v4: legacy architecture (default, for NLL training)
STEERNET_ARCH_ARGS = "--steernet_arch v5"
print(f"Architecture: {STEERNET_ARCH_ARGS}")

## 5b. Clear Stale Cache


In [ ]:
# ========== PHASE 1A→1B BUG-6 FIX: Clear Stale Cache ==========
# Cache fingerprint was missing κ and architecture, causing stale output reuse.
# This is now fixed in benchmark.py, but we clear the cache to be safe.
import shutil
import os

cache_dir = "./cache/benchmark_outputs"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"✅ Cleared stale cache: {cache_dir}")
    os.makedirs(cache_dir, exist_ok=True)
else:
    print(f"✅ Cache directory clean: {cache_dir}")
    os.makedirs(cache_dir, exist_ok=True)

## 6. Hyperparameter Configuration (Phase 1 V2 — Zero Heuristic Dials)


In [ ]:
# ==============================================================================
# V9 EXPERIMENT A: Big SteerNet + On-Policy
# ==============================================================================
# For Experiment B (STaR): set RUN_STAR=True, ON_POLICY=False

S1_OBJECTIVE = 'nll'
LOSS_MODE    = 'typed'
K_SCALE_MODE = 'sqrt'
STEER_SCALE  = 0.30
BATCH_SIZE   = 2
GRAD_ACCUM   = 16
MAX_TOK_LEN  = 512

# ── SteerNet: 4x bigger ──
RHO_MAX      = 0.30      # was 0.12
STEER_KAPPA  = 0.08
LORA_RANK    = 128       # was 32
LORA_ALPHA   = 128.0     # match rank
KL_WEIGHT    = 0.1
S1_LR        = 8e-5
S1_EPOCHS    = 5
S1_TRAIN_N   = 5000
S1_VAL_N     = 100
S1_ANSWER_WEIGHT      = 1.0
S1_ANSWER_REGION_FRAC = 0.30

# ── On-Policy (Experiment A) ──
ON_POLICY              = True
ON_POLICY_K            = 4
ON_POLICY_TEMP         = 0.7
ON_POLICY_FRAC         = 0.50
ON_POLICY_START        = 2
ON_POLICY_INTERVAL     = 1
ON_POLICY_GOLD_FALLBACK = True

# ── S1R STaR (Experiment B — set True to enable) ──
RUN_STAR       = False
STAR_K         = 4
STAR_TEMP      = 0.7
STAR_EPOCHS    = 2
STAR_LR        = 5e-5
STAR_ROUNDS    = 1
STAR_HINT_K    = 2
STAR_GEN_BATCH = 4

# ── Disabled ──
RUN_CONTRASTIVE = False

CALIB_SEL_N  = 100
BENCH_N_TEST = 300
BENCH_SHOTS  = '0 2 4'

import math
n_layers = 10
_eff_amp = RHO_MAX / math.sqrt(n_layers)
print(f'V9-A | rank={LORA_RANK}, {n_layers} layers, rho_max={RHO_MAX}')
print(f'Per-layer amplitude: {_eff_amp:.3f} ({_eff_amp*100:.1f}% of hidden state)')
print(f'On-policy: k={ON_POLICY_K}, temp={ON_POLICY_TEMP}, frac={ON_POLICY_FRAC}')
print(f'STaR: {"ENABLED" if RUN_STAR else "disabled"}')

In [ ]:
# ==============================================================================
# V7: Router and Gate DISABLED — SteerNet-only ablation
# ==============================================================================
RUN_ROUTER = False
RUN_GATE   = False
BENCH_ARMS = 'baseline,fixed'

print(f'Router: DISABLED | Gate: DISABLED')
print(f'Benchmark arms: {BENCH_ARMS}')

## 7. Fixed Layer Configuration (Phase 1 V2)


In [ ]:
# ==============================================================================
# V9: ALL 10 candidate layers — bigger SteerNet
# ==============================================================================
FIXED_LAYERS = [8, 11, 13, 15, 17, 19, 20, 21, 22, 24]
S_FIXED = ','.join(map(str, FIXED_LAYERS))

print('='*80)
print(f'FIXED LAYER CONFIGURATION (V9 — {len(FIXED_LAYERS)} Layers)')
print('='*80)
print(f'Active layers ({len(FIXED_LAYERS)}): {FIXED_LAYERS}')
import math
_eff_amp = RHO_MAX / math.sqrt(len(FIXED_LAYERS))
print(f'Per-layer amplitude: rho_max/sqrt({len(FIXED_LAYERS)}) = {_eff_amp:.4f} ({_eff_amp*100:.1f}%)')
print('='*80)

## 8. Stage S1: SteerNet Training (V2 Typed Loss)


In [ ]:
# Stage S1: SteerNet Training — format learning + on-policy STaR
import subprocess

command = f'''python scripts/run_ars_pipeline.py --stage S1 {STEERNET_ARCH_ARGS} \
    --objective {S1_OBJECTIVE} \
    --s1_epochs {S1_EPOCHS} \
    --s1_lr {S1_LR} \
    --s1_train_n {S1_TRAIN_N} \
    --s1_val_n {S1_VAL_N} \
    --batch_size {BATCH_SIZE} \
    --steer_scale {STEER_SCALE} \
    --s_fixed {S_FIXED} \
    --set steer_kappa={STEER_KAPPA} \
    --set rho_max={RHO_MAX} \
    --set k_scale_mode={K_SCALE_MODE} \
    --set grad_accum={GRAD_ACCUM} \
    --set max_tok_len={MAX_TOK_LEN} \
    --set loss_mode={LOSS_MODE} \
    --set lora_rank={LORA_RANK} \
    --set lora_alpha={LORA_ALPHA} \
    --set kl_weight={KL_WEIGHT} \
    --set diversify_prompts=true \
    --set on_policy={str(ON_POLICY).lower()} \
    --set on_policy_k={ON_POLICY_K} \
    --set on_policy_temperature={ON_POLICY_TEMP} \
    --set on_policy_frac={ON_POLICY_FRAC} \
    --set on_policy_start_epoch={ON_POLICY_START} \
    --set on_policy_interval={ON_POLICY_INTERVAL} \
    --set on_policy_gold_fallback={str(ON_POLICY_GOLD_FALLBACK).lower()} \
    --set s1_answer_weight={S1_ANSWER_WEIGHT} \
    --set s1_answer_region_frac={S1_ANSWER_REGION_FRAC} \
    --set candidate_layers={S_FIXED} \
    --set fixed_layers={S_FIXED}'''

process = subprocess.Popen(
    command, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="")
if process.wait() != 0:
    raise RuntimeError(
        f"S1 pipeline failed with exit code {process.returncode}; "
        "full runner output is shown above."
    )

import os, glob
ckpts = sorted(glob.glob("checkpoints/steernet_epoch*.pt"))
print(f'\nPer-epoch checkpoints saved: {[os.path.basename(c) for c in ckpts]}')
print('\n' + '='*80)
print('S1 Training Complete')
print('='*80)

## 8b. Stage S2: Phase 2 Router Training (Per-Question Layer Selection)

In [ ]:
if RUN_ROUTER:
    import subprocess

    command = f'''python scripts/run_ars_pipeline.py --stage S2 {STEERNET_ARCH_ARGS} \
        --s1_train_n {S1_TRAIN_N} \
        --s1_val_n  {S1_VAL_N} \
        --s_fixed   {S_FIXED} \
        --steer_scale {STEER_SCALE} \
        --set steer_kappa={STEER_KAPPA} \
        --set k_scale_mode={K_SCALE_MODE} \
        --set lora_rank={LORA_RANK} \
        --set lora_alpha={LORA_ALPHA} \
        --set phase2_epochs={PHASE2_EPOCHS} \
        --set phase2_lr={PHASE2_LR} \
        --set router_max_k={ROUTER_MAX_K} \
        --set router_min_k={ROUTER_MIN_K} \
        --set router_warmup_epochs={ROUTER_WARMUP} \
        --set router_beta_entropy_subset={ROUTER_BETA_ENTROPY_SUBSET}'''

    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        print(f"[warning] S2 pipeline exited with code {process.returncode}")

    import torch, json
    ckpt_path = "checkpoints/router.pt"
    if os.path.exists(ckpt_path):
        print('\n' + '='*80)
        print('✅ S2 Router Training Complete')
        print('='*80)
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        print(f"Router checkpoint keys: {list(ckpt.keys())}")
        print(f"fixed_layers: {ckpt.get('fixed_layers')}")
        print(f"router_min_k: {ckpt.get('router_min_k')}  router_max_k: {ckpt.get('router_max_k')}")
else:
    print("Router training skipped (RUN_ROUTER=False)")

## 8c. Stage S2B: Phase 2B Gate Training (Per-Token Usefulness Gate)

In [ ]:
if RUN_GATE:
    import subprocess

    command = f'''python scripts/run_ars_pipeline.py --stage S2B {STEERNET_ARCH_ARGS} \
        --s1_train_n {S1_TRAIN_N} \
        --s1_val_n  {S1_VAL_N} \
        --s_fixed   {S_FIXED} \
        --steer_scale {STEER_SCALE} \
        --set steer_kappa={STEER_KAPPA} \
        --set k_scale_mode={K_SCALE_MODE} \
        --set lora_rank={LORA_RANK} \
        --set lora_alpha={LORA_ALPHA} \
        --set use_gate=true \
        --set gate_epochs={GATE_EPOCHS} \
        --set gate_lr={GATE_LR} \
        --set gate_finetune_steernet={str(GATE_FINETUNE_STEERNET).lower()}'''

    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        print(f"[warning] S2B pipeline exited with code {process.returncode}")

    # Verify gate checkpoint
    import torch
    gate_path = "checkpoints/gate.pt"
    sn_gated_path = "checkpoints/steernet_gated.pt"
    if os.path.exists(gate_path):
        print('\n' + '='*80)
        print('✅ S2B Gate Training Complete')
        print('='*80)
        ckpt = torch.load(gate_path, map_location="cpu", weights_only=False)
        print(f"Gate checkpoint: best_val_nll={ckpt.get('best_val_nll', 'N/A'):.4f}")
        print(f"Fixed layers: {ckpt.get('fixed_layers')}")
        print(f"Gate epochs: {ckpt.get('gate_epochs')}")
    if os.path.exists(sn_gated_path):
        print(f"Gated SteerNet checkpoint: {sn_gated_path} ✅")
else:
    print("Gate training skipped (RUN_GATE=False)")

## 8d. Stage S1C: Contrastive Residual Training (Reasoning Direction)

**What it does:**
1. Generates greedy completions from the steered model for all training questions
2. Filters to questions where the model is WRONG (hard-example curriculum)
3. Computes the direction vector between correct and incorrect reasoning hidden states
4. Trains SteerNet delta to align with that direction (cosine similarity loss)

The contrastive loss teaches SteerNet WHERE to push in activation space for correct reasoning,
instead of just learning to mimic gold token sequences (which only learns format).

In [ ]:
if RUN_CONTRASTIVE:
    import subprocess

    command = f'''python scripts/run_ars_pipeline.py --stage S1C {STEERNET_ARCH_ARGS} \
        --s1_train_n {S1_TRAIN_N} \
        --s1_val_n {S1_VAL_N} \
        --batch_size {BATCH_SIZE} \
        --s_fixed {S_FIXED} \
        --steer_scale {STEER_SCALE} \
        --set steer_kappa={STEER_KAPPA} \
        --set rho_max={RHO_MAX} \
        --set k_scale_mode={K_SCALE_MODE} \
        --set lora_rank={LORA_RANK} \
        --set lora_alpha={LORA_ALPHA} \
        --set kl_weight={KL_WEIGHT} \
        --set loss_mode={LOSS_MODE} \
        --set grad_accum={GRAD_ACCUM} \
        --set max_tok_len={MAX_TOK_LEN} \
        --set contrastive_phase=true \
        --set contrastive_epochs={CONTRASTIVE_EPOCHS} \
        --set contrastive_lr={CONTRASTIVE_LR} \
        --set contrastive_weight={CONTRASTIVE_WEIGHT} \
        --set contrastive_nll_weight={CONTRASTIVE_NLL_W} \
        --set contrastive_frac={CONTRASTIVE_FRAC} \
        --set candidate_layers={S_FIXED} \
        --set fixed_layers={S_FIXED}'''

    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(
            f"S1C pipeline failed with exit code {process.returncode}; "
            "full runner output is shown above."
        )

    import os, glob
    ckpts = sorted(glob.glob("checkpoints/steernet_s1c_epoch*.pt"))
    print(f'\nS1C checkpoints: {[os.path.basename(c) for c in ckpts]}')
    print('\n' + '='*80)
    print('S1C Contrastive Training Complete')
    print('='*80)
else:
    print("Contrastive training skipped (RUN_CONTRASTIVE=False)")

## 8e. Stage S1R: Self-Taught Reasoning (STaR)

**What it does:**
1. Generates k=8 completions per question at temp=0.7 with steering ON
2. Scores each: keeps only **correct** completions (rejection sampling)
3. For hard questions (all 8 wrong): adds the gold answer as a **hint**, regenerates,
   then trains on the hint-generated solution WITHOUT the hint
4. Trains SteerNet with standard NLL on these self-generated correct solutions

**Why this works:** Training on gold CoT teaches FORMAT (the gold has specific wording).
Training on the model's OWN correct solutions teaches it to AMPLIFY reasoning paths
it already knows how to produce — much easier for SteerNet to learn.

In [ ]:
if RUN_STAR:
    import subprocess

    command = f'''python scripts/run_ars_pipeline.py --stage S1R {STEERNET_ARCH_ARGS} \
        --s1_train_n {S1_TRAIN_N} \
        --s1_val_n {S1_VAL_N} \
        --batch_size {BATCH_SIZE} \
        --s_fixed {S_FIXED} \
        --steer_scale {STEER_SCALE} \
        --set steer_kappa={STEER_KAPPA} \
        --set rho_max={RHO_MAX} \
        --set k_scale_mode={K_SCALE_MODE} \
        --set lora_rank={LORA_RANK} \
        --set lora_alpha={LORA_ALPHA} \
        --set kl_weight={KL_WEIGHT} \
        --set loss_mode={LOSS_MODE} \
        --set grad_accum={GRAD_ACCUM} \
        --set max_tok_len={MAX_TOK_LEN} \
        --set star_k={STAR_K} \
        --set star_temperature={STAR_TEMP} \
        --set star_epochs={STAR_EPOCHS} \
        --set star_lr={STAR_LR} \
        --set star_rounds={STAR_ROUNDS} \
        --set star_hint_k={STAR_HINT_K} \
        --set star_gen_batch={STAR_GEN_BATCH} \
        --set candidate_layers={S_FIXED} \
        --set fixed_layers={S_FIXED}'''

    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(
            f"S1R pipeline failed with exit code {process.returncode}; "
            "full runner output is shown above."
        )

    import os, glob
    ckpts = sorted(glob.glob("checkpoints/steernet_s1r_*.pt"))
    print(f'\nS1R checkpoints: {[os.path.basename(c) for c in ckpts]}')
    print('\n' + '='*80)
    print('S1R Self-Taught Reasoning Complete')
    print('='*80)
else:
    print("STaR training skipped (RUN_STAR=False)")

## 9. Stage S5: Deployment Calibration (Contract v3)


In [ ]:
# Stage S5: Calibration — writes deployment_contract.json (version 3)
!python scripts/run_ars_pipeline.py --stage S5 {STEERNET_ARCH_ARGS} \
    --calib_sel_n {CALIB_SEL_N} \
    --contract deployment_contract.json \
    --s_fixed {S_FIXED} \
    --steer_scale {STEER_SCALE} \
    --set steer_kappa={STEER_KAPPA} \
    --set rho_max={RHO_MAX} \
    --set k_scale_mode={K_SCALE_MODE} \
    --set lora_rank={LORA_RANK} \
    --set lora_alpha={LORA_ALPHA} \
    --set candidate_layers={S_FIXED} \
    --set fixed_layers={S_FIXED}

import json as _json, pathlib as _pl
_c = _json.loads(_pl.Path("deployment_contract.json").read_text())
print(f"\nContract version: {_c.get('version')}  (expected 3)")
assert _c.get("version") == 3, "Expected contract v3!"
print("\n" + "="*80)
print("✅ S5 Calibration Complete (Contract v3 verified)")
print("="*80)

## 10. Benchmark: Baseline vs Fixed-Control (Zero Heuristic Dials)


In [ ]:
# Benchmark: Baseline vs Fixed-control (SteerNet only — no Router/Gate)
!python scripts/run_ars_pipeline.py --stage bench {STEERNET_ARCH_ARGS} \
    --contract deployment_contract.json \
    --n_test {BENCH_N_TEST} \
    --shots {BENCH_SHOTS} \
    --record experiment_record.json \
    --report \
    --arms {BENCH_ARMS} \
    --s_fixed {S_FIXED} \
    --steer_scale {STEER_SCALE} \
    --set steer_kappa={STEER_KAPPA} \
    --set rho_max={RHO_MAX} \
    --set lora_rank={LORA_RANK} \
    --set lora_alpha={LORA_ALPHA} \
    --set candidate_layers={S_FIXED} \
    --set fixed_layers={S_FIXED}

print('\n' + '='*80)
print('Benchmark Complete')
print('='*80)

## 11. Inspect Benchmark Report


In [ ]:
import os, json
from IPython.display import display
from src.evaluation.diagnostics import plot_paired, plot_training, show_generations

# Report is now written to report/REPORT_GSM8K.md (suite-scoped) and report/REPORT.md (alias)
for report_path in ["report/REPORT_GSM8K.md", "report/REPORT.md"]:
    if os.path.exists(report_path):
        print(f"--- {report_path} ---")
        with open(report_path, "r", encoding="utf-8") as f:
            print(f.read())
        break
else:
    print("Report not found. Check benchmark logs above.")

if os.path.exists("report/training_log.json"):
    with open("report/training_log.json", encoding="utf-8") as f:
        hist = json.load(f)
    plot_training(hist, "report/training_curves.png")
    print(f"\nBest crit: {hist.get('best_crit', 'N/A')}")
    for ep_info in hist.get("epochs", []):
        if isinstance(ep_info, dict):
            print(f"  crit={ep_info.get('crit', '?'):.4f}  result_top1={ep_info.get('result_top1', '?'):.3f}  val_nll={ep_info.get('validation_nll', '?'):.4f}")

if os.path.exists("experiment_record.json"):
    with open("experiment_record.json", encoding="utf-8") as f:
        record = json.load(f)
    plot_paired(record, "report/paired_outcomes.png")
    print("\n✅ Acceptance Criteria Check:")
    shots_data = record.get("shots", {})
    for shot_str, shot_data in shots_data.items():
        b_acc  = shot_data.get("baseline", {}).get("lenient_accuracy", 0.0)
        s_acc  = shot_data.get("fixed",    {}).get("lenient_accuracy", 0.0)
        r_acc  = shot_data.get("router",   {}).get("lenient_accuracy", None)
        delta_fixed  = s_acc - b_acc
        delta_router = (r_acc - b_acc) if r_acc is not None else None
        paired = shot_data.get("paired", {})
        router_str = f" | Router={r_acc:.1f}% (Δ{delta_router:+.1f}pp)" if r_acc is not None else ""
        print(f"  {shot_str}-shot | Baseline={b_acc:.1f}% | Fixed={s_acc:.1f}% (Δ{delta_fixed:+.1f}pp){router_str} | Wins={paired.get('fixed_wins',0)} Losses={paired.get('fixed_losses',0)}")
        if shot_data.get("paired_router"):
            pr = shot_data["paired_router"]
            print(f"           Router wins={pr.get('fixed_wins',0)} losses={pr.get('fixed_losses',0)} p={pr.get('mcnemar_p_value',1.0):.4f}")
        if paired.get("taxonomy"):
            print(f"    Taxonomy: {paired['taxonomy']}")

# Per-arm generation files
for shot_str in shots_data.keys():
    gen_path = f"report/generations/gsm8k_{shot_str}shot_fixed.jsonl"
    if os.path.exists(gen_path):
        display(show_generations(gen_path, k=4, flips_only=False))

## 12. Save Artifacts


In [ ]:
import shutil, os, json, glob

os.makedirs("/kaggle/working/ars_artifacts", exist_ok=True)

# Core artifacts
for fname in [
    "deployment_contract.json",
    "experiment_record.json",
    "norms_calibration.json",
    "checkpoints/steernet.pt",
    "checkpoints/router.pt",
    "checkpoints/gate.pt",
    "checkpoints/steernet_gated.pt",
]:
    if os.path.exists(fname):
        shutil.copy(fname, "/kaggle/working/ars_artifacts/")
        print(f"Copied: {fname}")

# Per-epoch SteerNet checkpoints (for ablation)
for ckpt in sorted(glob.glob("checkpoints/steernet_epoch*.pt")):
    shutil.copy(ckpt, "/kaggle/working/ars_artifacts/")
    print(f"Copied: {ckpt}")

if os.path.exists("report"):
    shutil.copytree("report", "/kaggle/working/ars_artifacts/report", dirs_exist_ok=True)
    print("Copied: report/")

# Verify contract version in artifact
_c_path = "/kaggle/working/ars_artifacts/deployment_contract.json"
if os.path.exists(_c_path):
    _c = json.loads(open(_c_path).read())
    print(f"\nArtifact contract version: {_c.get('version')}")
    assert _c.get("version") == 3, "Contract must be v3 in artifact!"
    assert "shot_scales" not in _c, "shot_scales must not be in artifact contract!"
    print("✅ Contract v3 artifact verified")

print("\nArtifacts saved to /kaggle/working/ars_artifacts:")
!ls -lh /kaggle/working/ars_artifacts

## 13. Export Benchmark Bundle (ZIP)

In [ ]:
import zipfile, os, shutil
from pathlib import Path

# Files needed to run any benchmark on another machine
BUNDLE_FILES = [
    "deployment_contract.json",   # route + envelope metadata
    "norms_calibration.json",     # per-layer norm caps (required by SteerNet)
    "checkpoints/steernet.pt",    # Phase 1 SteerNet weights
    "checkpoints/router.pt",      # Phase 2 Router weights (if trained)
]

ZIP_PATH = "/kaggle/working/ars_benchmark_bundle.zip"

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in BUNDLE_FILES:
        if os.path.exists(fpath):
            zf.write(fpath)
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"  + {fpath}  ({size_mb:.1f} MB)")
        else:
            print(f"  - {fpath}  [NOT FOUND — skipped]")

total_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"\n✅ Bundle saved: {ZIP_PATH}  ({total_mb:.1f} MB)")
print("Download this single file to run benchmarks on any machine.")